# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dineshsinghdhami/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [2]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
);
""")

WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"

print("Warehouse connection ready.")

Warehouse connection ready.


In [3]:
table_path = f"{WAREHOUSE}/fact_content_daily_performance/**/*.parquet"

schema = con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet('{table_path}', hive_partitioning=true)
LIMIT 1
""").df()

schema

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [4]:
q1 = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) - COUNT(DISTINCT (report_date, client_hash_id, content_hash_id))
        AS duplicate_grain_rows
FROM read_parquet('{table_path}', hive_partitioning=true)
WHERE month = '2026-03'
""").df()

q1

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,duplicate_grain_rows
0,9841378,0


In [5]:
q2 = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date,
    COUNT(DISTINCT client_hash_id) AS clients,
    COUNT(DISTINCT content_hash_id) AS content_items
FROM read_parquet('{table_path}', hive_partitioning=true)
WHERE month = '2026-03'
""").df()

q2

,row_count,first_date,last_date,clients,content_items
0,9841378,2026-03-01,2026-03-31,55,331437


In [6]:
q3 = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
          AND ga4_data_available IS TRUE
    ) AS both_available_rows
FROM read_parquet('{table_path}', hive_partitioning=true)
WHERE month = '2026-03'
""").df()

q3

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,ga4_available_rows,both_available_rows
0,9841378,3611061,413966,364347


### Unit of analysis + time window

For this lane, one row represents the daily performance of one pseudonymized content item for one pseudonymized client on one report date.

The main table I use is `fact_content_daily_performance`.

For development and verification, I use March 2026 (`month = '2026-03'`), covering 2026-03-01 through 2026-03-31.

The March slice contains 9,841,378 rows across 55 pseudonymized clients and 331,437 content items. The grain check found 0 duplicate combinations of `report_date`, `client_hash_id`, and `content_hash_id`, which supports the stated row grain.

My lane is to rank content items by content-refresh priority so that higher-priority pages can be reviewed first.

I deliberately exclude future information from the feature set because information that is only known after the decision point would create label leakage.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Field contract

**Features**
- `gsc_impressions` — search visibility available at the decision moment.
- `gsc_clicks` — observed organic search traffic.
- `gsc_avg_position` — observed average search position.
- `ga4_pageviews` — observed page-level traffic.
- `ga4_engaged_sessions` — observed engagement signal.

**Label / proxy**
- A derived content-refresh priority outcome used to rank pages for review. The label must be computed from the chosen outcome window and must not be included as an input feature.

**Context / identifiers**
- `report_date`
- `client_hash_id`
- `content_hash_id`
- `month`
- `gsc_data_available`
- `ga4_data_available`

The availability fields are used to decide whether the corresponding source data is usable; they are not treated as performance measurements.

**Excluded**
- Any future-period outcome or feature derived from the label, because it would reveal information that was not available at the decision moment.
- Raw identifiers are retained only for joins and grouping, not as predictive signals.

In [7]:
features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_engaged_sessions",
]

label_proxy = "future_content_refresh_need"

context_fields = [
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "month",
    "gsc_data_available",
    "ga4_data_available",
]

excluded_fields = [
    "future-period outcomes",
    "label-derived features",
    "raw identifiers as predictive features",
]

print("Features:", features)
print("Label / proxy:", label_proxy)
print("Context fields:", context_fields)
print("Excluded:", excluded_fields)

Features: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_engaged_sessions']
Label / proxy: future_content_refresh_need
Context fields: ['report_date', 'client_hash_id', 'content_hash_id', 'month', 'gsc_data_available', 'ga4_data_available']
Excluded: ['future-period outcomes', 'label-derived features', 'raw identifiers as predictive features']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Five-feature frame

The feature snapshot uses March 2026 information only.

- `march_impressions` — knowable at the decision moment because it is aggregated from March GSC impressions.
- `march_clicks` — knowable at the decision moment because it is aggregated from March GSC clicks.
- `march_avg_position` — knowable at the decision moment because it summarizes observed March search position.
- `march_pageviews` — knowable at the decision moment because it is aggregated from March GA4 pageviews.
- `march_engaged_sessions` — knowable at the decision moment because it is aggregated from March GA4 engaged sessions.

The label is based on a later outcome window. April information is used only to create the outcome and is not part of the honest March feature set.

In [8]:
feature_frame = con.sql(f"""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS march_impressions,
        SUM(gsc_clicks) AS march_clicks,
        AVG(gsc_avg_position) AS march_avg_position,
        SUM(ga4_pageviews) AS march_pageviews,
        SUM(ga4_engaged_sessions) AS march_engaged_sessions
    FROM read_parquet('{table_path}', hive_partitioning=true)
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
      AND ga4_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
),

april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS april_clicks
    FROM read_parquet('{table_path}', hive_partitioning=true)
    WHERE month = '2026-04'
      AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)

SELECT
    m.client_hash_id,
    m.content_hash_id,
    m.march_impressions,
    m.march_clicks,
    m.march_avg_position,
    m.march_pageviews,
    m.march_engaged_sessions,
    a.april_clicks,

    CASE
        WHEN a.april_clicks < m.march_clicks * 0.80 THEN 1
        ELSE 0
    END AS needs_refresh

FROM march m
JOIN april a
  ON m.client_hash_id = a.client_hash_id
 AND m.content_hash_id = a.content_hash_id
WHERE m.march_clicks > 0
""").df()

print("Feature frame shape:", feature_frame.shape)

feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (38962, 9)


,client_hash_id,content_hash_id,march_impressions,march_clicks,march_avg_position,march_pageviews,march_engaged_sessions,april_clicks,needs_refresh
0,client_9958f0a7ae1df715,content_88180b9798e157df,44.0,1.0,23.432292,3.0,0.0,1.0,0
1,client_9958f0a7ae1df715,content_3c3b575d53a71932,1636.0,1.0,9.702400,26.0,0.0,0.0,1
2,client_9958f0a7ae1df715,content_757d8500451fb901,72.0,1.0,12.112355,2.0,0.0,1.0,0
3,client_9958f0a7ae1df715,content_d18b0265046ac49a,303.0,3.0,6.198434,15.0,1.0,0.0,1
4,client_9958f0a7ae1df715,content_23de62a247d3f2f4,35.0,5.0,15.230000,7.0,0.0,2.0,1


### Deliberate leakage experiment

To demonstrate label leakage, I intentionally create one feature directly from the target. This feature would never be available at the decision moment because it is derived from the outcome itself.

I first include it and record the suspiciously strong model score. Then I remove it and train the same type of model using only the five March features. The second score is the honest result.

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

honest_features = [
    "march_impressions",
    "march_clicks",
    "march_avg_position",
    "march_pageviews",
    "march_engaged_sessions",
]

y = feature_frame["needs_refresh"]

# --------------------------------------------------
# 1. INTENTIONAL LEAKAGE
# --------------------------------------------------

leak_frame = feature_frame.copy()

# Deliberately wrong: this column is derived directly from the label.
leak_frame["leaky_label_copy"] = leak_frame["needs_refresh"]

leaky_features = honest_features + ["leaky_label_copy"]

X_train, X_test, y_train, y_test = train_test_split(
    leak_frame[leaky_features],
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

leaky_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

leaky_model.fit(X_train, y_train)

leaky_pred = leaky_model.predict(X_test)
leaky_prob = leaky_model.predict_proba(X_test)[:, 1]

print("WITH intentional label leakage")
print("Accuracy:", round(accuracy_score(y_test, leaky_pred), 4))
print("ROC-AUC :", round(roc_auc_score(y_test, leaky_prob), 4))

WITH intentional label leakage
Accuracy: 1.0
ROC-AUC : 1.0


In [10]:
# --------------------------------------------------
# 2. REMOVE THE LEAK AND KEEP THE HONEST SCORE
# --------------------------------------------------

X = feature_frame[honest_features]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

honest_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

honest_model.fit(X_train, y_train)

honest_pred = honest_model.predict(X_test)
honest_prob = honest_model.predict_proba(X_test)[:, 1]

honest_accuracy = accuracy_score(y_test, honest_pred)
honest_auc = roc_auc_score(y_test, honest_prob)

print("AFTER removing leakage")
print("Accuracy:", round(honest_accuracy, 4))
print("ROC-AUC :", round(honest_auc, 4))

# The leaked feature is not part of the final feature set.
final_features = honest_features.copy()

print("\nFinal feature count:", len(final_features))
print("Final features:", final_features)

AFTER removing leakage
Accuracy: 0.6025
ROC-AUC : 0.5881

Final feature count: 5
Final features: ['march_impressions', 'march_clicks', 'march_avg_position', 'march_pageviews', 'march_engaged_sessions']


### Leakage lesson

When I intentionally included a feature derived directly from the target, the model achieved an Accuracy of **1.0000** and ROC-AUC of **1.0000**. This apparently perfect result is misleading because the model was given information derived from the answer it was supposed to predict.

After removing the leaked feature and keeping only the five March features available at the decision moment, Accuracy decreased to **0.6025** and ROC-AUC to **0.5881**.

I keep the lower score as the honest result. It reflects the information that would actually have been available when deciding which content items should be reviewed.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data limits

This slice has several limitations.

The target is moderately imbalanced: 59.93% of rows are labeled 0 and 40.07% are labeled 1, so evaluation should not rely on accuracy alone.

First, GSC and GA4 coverage is incomplete. In March 2026 there were 9,841,378 total daily rows, but only 3,611,061 had GSC data available, 4,139,366 had GA4 data available, and 364,347 had both available. The feature frame therefore represents only content for which the required signals are available.

Second, the label is a proxy rather than a direct human judgment of whether content actually needs refreshing. I define `needs_refresh` from a later decline in clicks, so it captures one measurable outcome rather than every reason a page might need attention.

Third, the development slice uses March features and an April outcome. Results from this time window may not generalize equally well to other months or clients because traffic patterns and data availability can change.

Finally, this analysis is decision-support only. The model score should help prioritize pages for review, not automatically decide that a page must be refreshed.

In [12]:
# Check the target balance in the modeling frame
label_summary = (
    feature_frame["needs_refresh"]
    .value_counts(dropna=False)
    .rename_axis("needs_refresh")
    .reset_index(name="rows")
)

label_summary["share"] = (
    label_summary["rows"] / len(feature_frame)
).round(4)

print("Modeling rows:", len(feature_frame))
label_summary

Modeling rows: 38962


,needs_refresh,rows,share
0,0,23350,0.5993
1,1,15612,0.4007


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

### Self-check

- The unit of analysis and March 2026 development window are stated.
- The row grain was verified with 0 duplicate grain rows.
- Exactly three warehouse verification queries were run and their outputs are visible.
- Availability was checked using `IS TRUE`.
- The five-feature frame contains only features knowable at the decision moment.
- A future outcome was used only as the label/proxy.
- Intentional label leakage produced a perfect but invalid score.
- The leaked feature was removed.
- The honest model score is retained: Accuracy = 0.6025 and ROC-AUC = 0.5881.
- At least one limitation of the data slice is documented.